# 2.7c — Comparer LIBSVM et SMO

[Source Python](2.7c-SVM-SOTA-Comparison.ipynb) | [Index](README.md)

**Objectifs** : mesurer les ajustements à froid, reconstruire le dual, calculer KKT et gap, étudier la tolérance. **Prérequis** : [SMO et dualité](2.7b-SMO-From-Scratch.ipynb). **Durée indicative** : 45 minutes.

Le vrai `sklearn.svm.SVC` utilise LIBSVM via Python.NET ; C# pilote les ajustements et calcule les diagnostics. Une autre bibliothèque SVM ne garantirait pas ici les mêmes `max_iter`, coefficients, supports et compteurs.

In [1]:
#r "nuget: PythonNet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Text.Json;
using System.Globalization;
using System.Diagnostics;
using Python.Runtime;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
// Le pont charge le solveur LIBSVM réel et conserve le tirage NumPy partagé.
if (!PythonEngine.IsInitialized)
{
    var library = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (string.IsNullOrWhiteSpace(library))
        throw new InvalidOperationException("Configurer PYTHONNET_PYDLL et installer NumPy et scikit-learn.");
    Runtime.PythonDLL = library;
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}
double[][] Matrix(JsonElement e) => e.EnumerateArray().Select(row => row.EnumerateArray().Select(v => v.GetDouble()).ToArray()).ToArray();
double[] Vector(JsonElement e) => e.EnumerateArray().Select(v => v.GetDouble()).ToArray();
int[] Integers(JsonElement e) => e.EnumerateArray().Select(v => v.GetInt32()).ToArray();
PyModule py;
using (Py.GIL())
{
    py = Py.CreateScope();
    py.Exec(@"import json, time, numpy as np, sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
X,y0=make_moons(n_samples=200,noise=.25,random_state=42)
y=np.where(y0==0,-1.,1.)
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=.30,random_state=42,stratify=y)
C,gamma,tol=1.,2.,1e-3
start=time.perf_counter(); clf=SVC(C=C,kernel='rbf',gamma=gamma,tol=tol,max_iter=20000).fit(X_tr,y_tr); duration=time.perf_counter()-start
xmin,xmax=float(X[:,0].min()-.5),float(X[:,0].max()+.5)
ymin,ymax=float(X[:,1].min()-.5),float(X[:,1].max()+.5)
xx,yy=np.meshgrid(np.linspace(xmin,xmax,200),np.linspace(ymin,ymax,200))
grid=np.c_[xx.ravel(),yy.ravel()]
seed=np.random.default_rng(7); Xw=seed.normal(size=(7,3)); yw=np.where(seed.random(7)>.5,1.,-1.); aw=np.round(seed.random(7)*.5,3)
result=json.dumps(dict(train=X_tr.tolist(),test=X_te.tolist(),ytrain=y_tr.tolist(),ytest=y_te.tolist(),support=clf.support_.tolist(),dual=clf.dual_coef_[0].tolist(),bias=float(clf.intercept_[0]),test_scores=clf.decision_function(X_te).tolist(),train_scores=clf.decision_function(X_tr).tolist(),xmin=xmin,xmax=xmax,ymin=ymin,ymax=ymax,duration=duration,xw=Xw.tolist(),yw=yw.tolist(),aw=aw.tolist(),numpy=np.__version__,sklearn=sklearn.__version__))");
}
JsonElement Data()
{
    using (Py.GIL()) return JsonDocument.Parse(py.Get("result").As<string>()).RootElement;
}
var data = Data();
var train = Matrix(data.GetProperty("train"));
var test = Matrix(data.GetProperty("test"));
var ytrain = Vector(data.GetProperty("ytrain"));
var ytest = Vector(data.GetProperty("ytest"));
var supportSk = Integers(data.GetProperty("support"));
var alphaSk = new double[ytrain.Length];
var signedSk = Vector(data.GetProperty("dual"));
for (int i = 0; i < supportSk.Length; i++) alphaSk[supportSk[i]] = Math.Abs(signedSk[i]);
Console.WriteLine($"NumPy {data.GetProperty("numpy").GetString()}, scikit-learn {data.GetProperty("sklearn").GetString()} ; train={train.Length}, test={test.Length}, C=1, gamma=2, tol=1e-3");

Installed Packages PythonNet, 3.0.5

NumPy 2.3.5, scikit-learn 1.8.0 ; train=140, test=60, C=1, gamma=2, tol=1e-3


## Données, SMO et mesures à froid

Les 200 points et le split stratifié 140/60 sont identiques au carnet précédent. Le premier réglage C=1, γ=2, tol=10⁻³ utilise `max_iter=20 000` ; 50 estimateurs neufs sont ajustés pour mesurer un temps moyen. Le temps C# inclut les appels Python.NET et ne doit pas être comparé à une mesure historique.

In [2]:
public record SmoModel(double[] Alpha, double Bias, int Examinations);
public static class Smo
{
    public static double Kernel(double[] x, double[] y, double gamma)
    {
        double distance = 0;
        for (int i = 0; i < x.Length; i++)
        {
            double difference = x[i] - y[i];
            distance += difference * difference;
        }
        return Math.Exp(-gamma * distance);
    }
    public static double[,] Gram(double[][] x, double gamma)
    {
        var result = new double[x.Length,x.Length];
        for (int i = 0; i < x.Length; i++)
            for (int j = 0; j < x.Length; j++)
                result[i,j] = Kernel(x[i], x[j], gamma);
        return result;
    }
    public static double Dual2(double aj, double c, double yi, double yj, double vi, double vj, double kii, double kjj, double kij)
    {
        double ai = yi * (c - aj * yj);
        return ai + aj - .5 * (kii * ai * ai + kjj * aj * aj + 2 * yi * yj * kij * ai * aj) - ai * yi * vi - aj * yj * vj;
    }
    public static bool Step(int i, int j, double[] y, double[] alpha, ref double bias, double[,] gram, double C, double eps, double[] errors)
    {
        if (i == j) return false;
        double ai = alpha[i], aj = alpha[j], yi = y[i], yj = y[j];
        double lower, upper;
        if (yi != yj)
        {
            lower = Math.Max(0, aj-ai);
            upper = Math.Min(C, C+aj-ai);
        }
        else
        {
            lower = Math.Max(0, ai+aj-C);
            upper = Math.Min(C, ai+aj);
        }
        if (lower >= upper) return false;
        double kii = gram[i,i], kjj = gram[j,j], kij = gram[i,j];
        double eta = kii + kjj - 2*kij;
        double next;
        if (eta > 0)
        {
            // Maximum quadratique non contraint, projeté dans l'intervalle admissible.
            next = Math.Clamp(aj + yj*(errors[i]-errors[j])/eta, lower, upper);
        }
        else
        {
            // Courbure nulle ou négative : comparer explicitement les deux bornes.
            double c = ai*yi + aj*yj;
            double vi = errors[i] + yi - bias - ai*yi*kii - aj*yj*kij;
            double vj = errors[j] + yj - bias - aj*yj*kjj - ai*yi*kij;
            double wLower = Dual2(lower,c,yi,yj,vi,vj,kii,kjj,kij);
            double wUpper = Dual2(upper,c,yi,yj,vi,vj,kii,kjj,kij);
            next = wLower > wUpper ? lower : wLower < wUpper ? upper : aj;
        }
        if (Math.Abs(next-aj) < eps*(aj+eps)) return false;
        double nextI = ai + yi*yj*(aj-next);
        alpha[i] = nextI;
        alpha[j] = next;
        double b1 = bias-errors[i]-yi*(nextI-ai)*kii-yj*(next-aj)*kij;
        double b2 = bias-errors[j]-yi*(nextI-ai)*kij-yj*(next-aj)*kjj;
        double nextBias = 0 < nextI && nextI < C ? b1 : 0 < next && next < C ? b2 : .5*(b1+b2);
        for (int k = 0; k < y.Length; k++)
            errors[k] += yi*(nextI-ai)*gram[k,i] + yj*(next-aj)*gram[k,j] + nextBias-bias;
        bias = nextBias;
        return true;
    }
    public static SmoModel Fit(double[][] x, double[] y, double C, double gamma, double tol = 1e-3, int maxPasses = 20, double eps = 1e-3)
    {
        int n = y.Length;
        var gram = Gram(x,gamma);
        var alpha = new double[n];
        var errors = y.Select(label => -label).ToArray();
        double bias = 0;
        int passes = 0, examinations = 0;
        while (passes < maxPasses)
        {
            int changes = 0;
            for (int i = 0; i < n; i++)
            {
                examinations++;
                double residual = y[i]*errors[i];
                if (!((residual < -tol && alpha[i] < C) || (residual > tol && alpha[i] > 0))) continue;
                // Premier candidat parmi les multiplicateurs strictement intérieurs.
                int selected = -1;
                double largest = -1;
                for (int j = 0; j < n; j++)
                {
                    if (j == i || alpha[j] <= 0 || alpha[j] >= C) continue;
                    double difference = Math.Abs(errors[i]-errors[j]);
                    if (difference > largest) { largest=difference; selected=j; }
                }
                bool changed = selected >= 0 && Step(i,selected,y,alpha,ref bias,gram,C,eps,errors);
                // Si la première paire échoue, balayer toutes les autres.
                if (!changed)
                    for (int j = 0; j < n; j++)
                        if (j != i && Step(i,j,y,alpha,ref bias,gram,C,eps,errors)) { changed=true; break; }
                if (changed) changes++;
            }
            passes = changes == 0 ? passes + 1 : 0;
        }
        return new SmoModel(alpha,bias,examinations);
    }
    public static double Score(double[] point, double[][] train, double[] labels, double[] alpha, double bias, double gamma)
    {
        double score = bias;
        for (int i = 0; i < train.Length; i++)
            if (alpha[i] > 1e-6) score += alpha[i]*labels[i]*Kernel(point,train[i],gamma);
        return score;
    }
    public static (double dual,double primal,double violation,double worst,double equality) Diagnostics(double[][] x, double[] y, double[] alpha, double bias, double C, double gamma)
    {
        var gram=Gram(x,gamma);
        double w2=0, sum=0, hinge=0, violation=0, worst=0, equality=0;
        for (int i=0;i<y.Length;i++)
        {
            sum+=alpha[i]; equality+=alpha[i]*y[i];
            for (int j=0;j<y.Length;j++) w2+=alpha[i]*y[i]*gram[i,j]*alpha[j]*y[j];
            double margin=y[i]*Score(x[i],x,y,alpha,bias,gamma);
            hinge+=Math.Max(0,1-margin);
            double low=alpha[i]<C-1e-12 ? Math.Max(0,1-margin):0;
            double high=alpha[i]>1e-12 ? Math.Max(0,margin-1):0;
            violation+=low+high;
            worst=Math.Max(worst,Math.Max(low,high));
        }
        return (sum-.5*w2,.5*w2+C*hinge,violation,worst,equality);
    }
}
Console.WriteLine("Noyau RBF, mise à jour par paires, SMO et diagnostics définis en C#.");


Noyau RBF, mise à jour par paires, SMO et diagnostics définis en C#.


Les primitives RBF, SMO et diagnostics sont définies ; chaque compteur est conservé dans son unité.

In [3]:
(JsonElement details, double seconds) FitLibsvm(double tolerance, int maxIterations, int repetitions)
{
    var timer=Stopwatch.StartNew();
    using (Py.GIL())
    {
        py.Set("test_tol",new PyFloat(tolerance));
        py.Set("test_max",new PyInt(maxIterations));
        // Un nouvel estimateur à chaque ajustement ; le chronomètre inclut le pont Python.NET.
        for(int i=0;i<repetitions;i++)
            py.Exec("SVC(C=1.,kernel='rbf',gamma=2.,tol=test_tol,max_iter=test_max).fit(X_tr,y_tr)");
    }
    timer.Stop();
    using (Py.GIL())
    {
        py.Exec(@"current=SVC(C=1.,kernel='rbf',gamma=2.,tol=test_tol,max_iter=test_max).fit(X_tr,y_tr)
result_current=json.dumps(dict(support=current.support_.tolist(),dual=current.dual_coef_[0].tolist(),bias=float(current.intercept_[0]),train_scores=current.decision_function(X_tr).tolist(),test_scores=current.decision_function(X_te).tolist(),iterations=int(current.n_iter_[0]),status=int(current.fit_status_)))");
        return (JsonDocument.Parse(py.Get("result_current").As<string>()).RootElement,timer.Elapsed.TotalSeconds/repetitions);
    }
}
(double[] alpha,double bias,double accuracy,int count,int iterations,int status) Unpack(JsonElement details)
{
    var indices=Integers(details.GetProperty("support"));
    var signed=Vector(details.GetProperty("dual"));
    var alpha=new double[train.Length];
    for(int i=0;i<indices.Length;i++) alpha[indices[i]]=Math.Abs(signed[i]);
    var predicted=Vector(details.GetProperty("test_scores"));
    double accuracy=predicted.Select((v,i)=>Math.Sign(v)==ytest[i]?1.0:0.0).Average();
    return (alpha,details.GetProperty("bias").GetDouble(),accuracy,indices.Length,details.GetProperty("iterations").GetInt32(),details.GetProperty("status").GetInt32());
}
Console.WriteLine("Chaque mesure crée un nouveau SVC ; diagnostics calculés en C#.");

Chaque mesure crée un nouveau SVC ; diagnostics calculés en C#.


## Résultats et dual reconstruit

`support_` et les valeurs absolues de `dual_coef_` reconstruisent le vecteur complet α. Les objectifs primal et dual, l'égalité et les violations KKT sont calculés à partir du modèle réellement ajusté.

In [4]:
// Reconstruire les diagnostics sur les modèles entraînés, sans comparer leurs unités de compteur.
var measure=FitLibsvm(1e-3,20000,50);
var lib=Unpack(measure.details);
var smoTimer=Stopwatch.StartNew();
var local=Smo.Fit(train,ytrain,1,2,1e-3,20,1e-3);
smoTimer.Stop();
var localScores=test.Select(row=>Smo.Score(row,train,ytrain,local.Alpha,local.Bias,2)).ToArray();
double localAccuracy=localScores.Select((v,i)=>Math.Sign(v)==ytest[i]?1.0:0.0).Average();
var libDiag=Smo.Diagnostics(train,ytrain,lib.alpha,lib.bias,1,2);
var localDiag=Smo.Diagnostics(train,ytrain,local.Alpha,local.Bias,1,2);
Console.WriteLine($"LIBSVM : accuracy={lib.accuracy:F3}, supports={lib.count}, temps moyen={measure.seconds*1000:F3}ms, itérations={lib.iterations}, statut={lib.status}");
Console.WriteLine($"SMO C# : accuracy={localAccuracy:F3}, supports={local.Alpha.Count(v=>v>1e-6)}, temps={smoTimer.Elapsed.TotalSeconds:F4}s, examens={local.Examinations}");
Console.WriteLine($"LIBSVM : dual={libDiag.dual:F6}, primal={libDiag.primal:F6}, gap={libDiag.primal-libDiag.dual:E3}, égalité={libDiag.equality:E3}");
Console.WriteLine($"LIBSVM KKT : somme={libDiag.violation:E3}, pire={libDiag.worst:E3} ; alpha zéro={lib.alpha.Count(v=>v<=1e-6)}, intérieur={lib.alpha.Count(v=>v>1e-6 && v<1-1e-6)}, borne={lib.alpha.Count(v=>v>=1-1e-6)}");
Console.WriteLine($"SMO : dual={localDiag.dual:F6}, gap={localDiag.primal-localDiag.dual:E3}, somme KKT={localDiag.violation:E3}");

LIBSVM : accuracy=0.967, supports=48, temps moyen=2.708ms, itérations=102, statut=0


SMO C# : accuracy=0.967, supports=48, temps=0.0072s, examens=6580


LIBSVM : dual=30.396089, primal=30.398078, gap=1.989E-003, égalité=4.441E-016


LIBSVM KKT : somme=4.115E-003, pire=5.224E-004 ; alpha zéro=92, intérieur=16, borne=32


SMO : dual=30.396087, gap=3.671E-003, somme KKT=5.582E-003


### Exercice 1 — Faire varier C

Entraîner SVC pour C=0,01, 0,1, 1, 10 et 100 avec γ=2 et tol=10⁻³ ; renvoyer supports et accuracy puis commenter les résultats observés, sans supposer une monotonie générale.

In [5]:
(int[] supports,double[] accuracy) ExerciceC(double[] values)
{
    // TODO : créer un SVC par C et mesurer supports et accuracy.
    return (null,null);
}
var valuesC=new[]{.01,.1,1.0,10.0,100.0};
var result=ExerciceC(valuesC);
if(result.supports==null && result.accuracy==null)
{
    Console.WriteLine("Exercice 1 à compléter.");
}
else if(result.supports==null || result.accuracy==null)
{
    Console.WriteLine("Exercice 1 : à revoir");
}
else
{
    bool ok=result.supports.Length==valuesC.Length && result.accuracy.Length==valuesC.Length;
    for(int k=0;ok && k<valuesC.Length;k++)
    {
        using(Py.GIL())
        {
            py.Set("exercise_c",new PyFloat(valuesC[k]));
            py.Exec("probe=SVC(C=exercise_c,kernel='rbf',gamma=2.,tol=1e-3).fit(X_tr,y_tr); probe_count=int(probe.n_support_.sum()); probe_accuracy=float((probe.predict(X_te)==y_te).mean())");
            ok=result.supports[k]==py.Get("probe_count").As<int>() && Math.Abs(result.accuracy[k]-py.Get("probe_accuracy").As<double>())<1e-10;
        }
    }
    Console.WriteLine(ok ? "Exercice 1 : correct" : "Exercice 1 : à revoir");
}

Exercice 1 à compléter.


## Tolérance de LIBSVM

Pour chaque tolérance, on effectue 50 ajustements froids avec `max_iter=200 000`, puis un ajustement supplémentaire dont les coefficients servent au diagnostic. Les écarts reflètent à la fois la tolérance, le calcul flottant et le protocole de mesure.

In [6]:
foreach(double tolerance in new[]{1e-2,1e-3,1e-4,1e-5})
{
    var run=FitLibsvm(tolerance,200000,50);
    var current=Unpack(run.details);
    var diagnostic=Smo.Diagnostics(train,ytrain,current.alpha,current.bias,1,2);
    Console.WriteLine($"tol={tolerance:E0}, temps moyen={run.seconds*1000:F3}ms, accuracy={current.accuracy:F3}, supports={current.count}, dual={diagnostic.dual:F6}, gap={diagnostic.primal-diagnostic.dual:E3}, itérations={current.iterations}, statut={current.status}");
}

tol=1E-002, temps moyen=3.177ms, accuracy=0.967, supports=49, dual=30.395293, gap=1.601E-002, itérations=69, statut=0


tol=1E-003, temps moyen=4.118ms, accuracy=0.967, supports=48, dual=30.396089, gap=1.989E-003, itérations=102, statut=0


tol=1E-004, temps moyen=4.273ms, accuracy=0.967, supports=48, dual=30.396094, gap=1.700E-004, itérations=134, statut=0


tol=1E-005, temps moyen=6.545ms, accuracy=0.967, supports=48, dual=30.396094, gap=2.361E-005, itérations=168, statut=0


### Exercice 2 — Reconstruire les scores

Recalculer chaque score de test à partir de α, y, noyau RBF et biais ; renvoyer l'écart maximum avec `decision_function` de SVC.

In [7]:
double[] ExerciceReconstruction(double[][] points,double[][] x,double[] y,double[] alpha,double bias,double gamma)
{
    // TODO : évaluer le noyau, sommer αᵢ yᵢ K puis ajouter le biais.
    return null;
}
var result=ExerciceReconstruction(test,train,ytrain,lib.alpha,lib.bias,2);
if(result==null)
{
    Console.WriteLine("Exercice 2 à compléter.");
}
else
{
    var reference=Vector(measure.details.GetProperty("test_scores"));
    bool ok=result.Length==reference.Length && result.Zip(reference,(a,b)=>Math.Abs(a-b)<1e-9).All(v=>v);
    var small=ExerciceReconstruction(new[]{new[]{0.0},new[]{1.0}},new[]{new[]{0.0},new[]{2.0}},new[]{1.0,-1.0},new[]{.7,.4},.2,.5);
    var expected=new[]{.7-.4*Math.Exp(-2)+.2,.7*Math.Exp(-.5)-.4*Math.Exp(-.5)+.2};
    ok=ok && small!=null && small.Length==2 && small.Zip(expected,(a,b)=>Math.Abs(a-b)<1e-9).All(v=>v);
    Console.WriteLine(ok ? "Exercice 2 : correct" : "Exercice 2 : à revoir");
}

Exercice 2 à compléter.


### Exercice 3 — Grille de validation croisée

Utiliser le vrai `GridSearchCV` avec C=[0,1 ; 1 ; 10] et γ=[0,5 ; 2 ; 8], cinq plis, scoring accuracy, puis rendre meilleurs paramètres, score de validation et accuracy test.

In [8]:
(double[] scores,double? bestC,double? bestGamma,double? bestCv,double? testAccuracy) ExerciceGrille()
{
    // TODO : utiliser GridSearchCV(SVC RBF, grille 3×3, cv=5), puis renvoyer scores et meilleur modèle.
    return (null,null,null,null,null);
}
var result=ExerciceGrille();
if(result.scores==null && result.bestC==null && result.bestGamma==null && result.bestCv==null && result.testAccuracy==null)
{
    Console.WriteLine("Exercice 3 à compléter.");
}
else if(result.scores==null || result.bestC==null || result.bestGamma==null || result.bestCv==null || result.testAccuracy==null)
{
    Console.WriteLine("Exercice 3 : à revoir");
}
else
{
    JsonElement reference;
    using(Py.GIL())
    {
        py.Exec("from sklearn.model_selection import GridSearchCV; g=GridSearchCV(SVC(kernel='rbf'),{'C':[.1,1.,10.],'gamma':[.5,2.,8.]},cv=5,scoring='accuracy').fit(X_tr,y_tr); grid_reference=json.dumps(dict(scores=g.cv_results_['mean_test_score'].tolist(),C=float(g.best_params_['C']),gamma=float(g.best_params_['gamma']),cv=float(g.best_score_),test=float((g.predict(X_te)==y_te).mean())))");
        reference=JsonDocument.Parse(py.Get("grid_reference").As<string>()).RootElement;
    }
    var expected=Vector(reference.GetProperty("scores"));
    bool ok=result.scores.Length==9 && result.scores.Zip(expected,(a,b)=>Math.Abs(a-b)<1e-10).All(v=>v) &&
        Math.Abs(result.bestC.Value-reference.GetProperty("C").GetDouble())<1e-10 &&
        Math.Abs(result.bestGamma.Value-reference.GetProperty("gamma").GetDouble())<1e-10 &&
        Math.Abs(result.bestCv.Value-reference.GetProperty("cv").GetDouble())<1e-10 &&
        Math.Abs(result.testAccuracy.Value-reference.GetProperty("test").GetDouble())<1e-10;
    Console.WriteLine(ok ? "Exercice 3 : correct" : "Exercice 3 : à revoir");
}

Exercice 3 à compléter.


## À retenir

Les coefficients des supports permettent de reconstruire le dual et de calculer les mêmes diagnostics que pour la SMO. Temps et compteurs décrivent des protocoles différents et ne sont pas interchangeables.

## Vérifiez votre compréhension

Pourquoi répéter des ajustements à froid ?

<details><summary>Réponse</summary>Pour rendre plus lisible un coût unitaire court, avec un nouvel estimateur à chaque répétition.</details>

## Pour aller plus loin

[SMO C#](2.7b-SMO-From-Scratch-CSharp.ipynb) et [source Python](2.7c-SVM-SOTA-Comparison.ipynb).